# 01 · Pre-game features

One row per game, every feature **as of the morning of that game**. Used by every later
EDA notebook (`%run ./01_pregame_features.ipynb`) and, later, by the model.

**Rules**
- **Point in time:** a team's numbers for a game use only its games on **earlier dates**.
  Nothing from that game, a later one, or another game the same day leaks in.
- **D-I vs D-I games only.** CHN records only partial stats for non-D-I opponents.
- **One row per game**, from the home team's view (at neutral sites, the team CHN lists as
  home). Each `diff_` feature is home minus away.
- **Target:** `home_win` = 1 or 0. Ties stay in the table (as NaN) so they can be counted;
  `model_rows` drops them.

**Windows** (per team, within the season), split by the tiers from step 4:
- **Tier 1** (shots, goal differential, win %): `season`, `last3`, `last5`, `last10`, and
  `ewm`, an exponentially weighted average in which a game's weight halves every 5 games.
- **Tier 2** (special teams, save %, discipline, game state, schedule): `season` only. These
  stats need 20–40 games to mean anything.
- Dropped as luck in step 4: shooting %, PDO and one-goal record.

No xG and no faceoffs (see `docs/eda_plan.md`).

> Outputs are stripped from git by nbstripout, so CHN data never lands in the repo.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable whether the notebook runs from notebooks/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import numpy as np
import pandas as pd
import psycopg

from config.settings import settings

SEASON = "2025-26"
WINDOWS = {"season": None, "last3": 3, "last5": 5, "last10": 10, "ewm": "ewm"}
EWM_HALFLIFE = 5  # games

# Raw per-game counts every rate is built from
COUNTS = [
    "games", "win", "goals_for", "goals_against", "en_goals", "shots_for", "shots_against",
    "close_sf", "close_sa", "pp_goals", "pp_opportunities", "pp_goals_against",
    "times_shorthanded", "pim", "scored_first", "first_goal_game", "p3_gf", "p3_ga",
]

TIER1 = ["win_pct", "goal_diff_pg", "goal_diff_pg_no_en", "shot_share", "close_shot_share",
         "shots_for_pg", "shots_against_pg"]
TIER2 = ["pp_pct", "pk_pct", "st_index", "pp_chances_pg", "tsh_pg", "pim_pg", "save_pct",
         "scored_first_pct", "p3_gd_pg"]
EXTRA = ["sos", "adj_goal_diff_pg", "starter_save_pct", "streak", "form_delta"]  # season-based / special

## Load games

One row per team per played D-I game, plus the goal list (for close-game shots, the first
goal and 3rd-period goals) and the goalie lines (for the likely starter).

In [ ]:
TEAM_GAMES_SQL = """
SELECT t.season, t.game_id, t.game_date, t.team_id, t.opponent_team_id, t.is_home, t.is_neutral,
       t.result, t.goals_for, t.goals_against, t.shots_for, t.shots_against,
       t.pp_goals, t.pp_opportunities, t.pp_goals_against, t.times_shorthanded,
       t.en_goals, t.pim, s.shots_p1, s.shots_p2, s.shots_p3
FROM features.team_games t
JOIN hockey.team_game_stats s USING (game_id, team_id)
JOIN hockey.teams me  ON me.team_id  = t.team_id
JOIN hockey.teams opp ON opp.team_id = t.opponent_team_id
WHERE t.season = %s
  AND me.is_d1 AND opp.is_d1  -- CHN records only partial stats for non-D-I teams
ORDER BY t.game_date, t.game_id
"""
GOALS_SQL = """
SELECT go.game_id, go.goal_number, go.team_id, go.period
FROM hockey.goals go JOIN hockey.games g USING (game_id)
WHERE g.season = %s AND g.is_played
"""
GOALIES_SQL = """
SELECT game_id, game_date, team_id, player_id, is_starter, saves, shots_against
FROM features.goalie_games WHERE season = %s
"""


def query(conn, sql, params):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


def add_ingredients(tg: pd.DataFrame, goals: pd.DataFrame) -> pd.DataFrame:
    """Per team-game counts that need the goal list: close-game shots, first goal, 3rd period."""
    tg = tg.copy()
    key = pd.MultiIndex.from_frame(tg[["game_id", "team_id"]])
    opp_key = pd.MultiIndex.from_frame(tg[["game_id", "opponent_team_id"]])
    shots = tg.set_index(["game_id", "team_id"])[["shots_p1", "shots_p2", "shots_p3"]]
    tg[["opp_p1", "opp_p2", "opp_p3"]] = shots.reindex(opp_key).to_numpy()

    per = goals.groupby(["game_id", "team_id", "period"]).size()

    def goals_in(lo, hi):
        s = per[(per.index.get_level_values("period") >= lo) & (per.index.get_level_values("period") <= hi)]
        s = s.groupby(level=["game_id", "team_id"]).sum()
        return s.reindex(key, fill_value=0).to_numpy(), s.reindex(opp_key, fill_value=0).to_numpy()

    m1, t1 = goals_in(1, 1)
    m2, t2 = goals_in(1, 2)
    close_p2 = np.abs(m1 - t1) <= 1  # the 2nd period started within one goal
    close_p3 = np.abs(m2 - t2) <= 1
    tg["close_sf"] = tg["shots_p1"] + tg["shots_p2"] * close_p2 + tg["shots_p3"] * close_p3
    tg["close_sa"] = tg["opp_p1"] + tg["opp_p2"] * close_p2 + tg["opp_p3"] * close_p3
    tg["p3_gf"], tg["p3_ga"] = goals_in(3, 3)

    first = goals.sort_values("goal_number").groupby("game_id")["team_id"].first()
    tg["first_goal_game"] = tg["game_id"].isin(first.index).astype(int)
    tg["scored_first"] = (tg["game_id"].map(first) == tg["team_id"]).astype(int)
    return tg.drop(columns=["opp_p1", "opp_p2", "opp_p3"])


def load(conn, season: str):
    tg = query(conn, TEAM_GAMES_SQL, (season,))
    goals = query(conn, GOALS_SQL, (season,))
    goalies = query(conn, GOALIES_SQL, (season,))
    tg["game_date"] = pd.to_datetime(tg["game_date"])
    goalies["game_date"] = pd.to_datetime(goalies["game_date"])
    goals = goals[goals["game_id"].isin(tg["game_id"])]
    goalies = goalies[goalies["game_id"].isin(tg["game_id"])]
    return add_ingredients(tg, goals), goalies

## Team state after each game date

For every team and window, running totals after each of its game dates, turned into rates.
Several games on one date are combined, so a lookup "before date D" never sees part of a
day.

The `ewm` window takes an exponentially weighted average of each count. Rates are ratios of
two such averages, so they stay proper rates (weighted goals ÷ weighted shots).

In [ ]:
def _ratio(num, den):
    return (num / den.where(den > 0)).astype(float)


def rates(t: pd.DataFrame, w: str) -> pd.DataFrame:
    """Summed (or weighted) counts for window `w` -> per-team rates."""
    c = lambda name: t[f"{name}_{w}"]
    g = c("games")
    out = {
        f"gp_{w}": g,
        f"win_pct_{w}": _ratio(c("win"), g),  # ties count half
        f"goal_diff_pg_{w}": _ratio(c("goals_for") - c("goals_against"), g),
        f"goal_diff_pg_no_en_{w}": _ratio(c("goals_for") - c("en_goals") - c("goals_against"), g),
        f"shot_share_{w}": _ratio(c("shots_for"), c("shots_for") + c("shots_against")),
        f"close_shot_share_{w}": _ratio(c("close_sf"), c("close_sf") + c("close_sa")),
        f"shots_for_pg_{w}": _ratio(c("shots_for"), g),
        f"shots_against_pg_{w}": _ratio(c("shots_against"), g),
    }
    if w == "season":  # tier 2: season only
        pp = _ratio(c("pp_goals"), c("pp_opportunities"))
        pk = 1 - _ratio(c("pp_goals_against"), c("times_shorthanded"))
        out |= {
            "pp_pct_season": pp,
            "pk_pct_season": pk,
            "st_index_season": pp + pk,
            "pp_chances_pg_season": _ratio(c("pp_opportunities"), g),
            "tsh_pg_season": _ratio(c("times_shorthanded"), g),
            "pim_pg_season": _ratio(c("pim"), g),
            "save_pct_season": 1 - _ratio(c("goals_against"), c("shots_against")),
            "scored_first_pct_season": _ratio(c("scored_first"), c("first_goal_game")),
            "p3_gd_pg_season": _ratio(c("p3_gf") - c("p3_ga"), g),
        }
    return pd.DataFrame(out, index=t.index)


def streak_after(results: pd.Series) -> pd.Series:
    """Signed streak after each game: W3 -> +3, L2 -> -2, a tie -> 0."""
    out, cur = [], 0
    for r in results:
        if r == "W":
            cur = cur + 1 if cur > 0 else 1
        elif r == "L":
            cur = cur - 1 if cur < 0 else -1
        else:
            cur = 0
        out.append(cur)
    return pd.Series(out, index=results.index)


def team_state(team_games: pd.DataFrame) -> pd.DataFrame:
    df = team_games.copy()
    df["win"] = (df["result"] == "W").astype(float) + 0.5 * (df["result"] == "T")
    df["games"] = 1
    df = df.sort_values(["team_id", "game_date", "game_id"])
    daily = df.groupby(["team_id", "game_date"], as_index=False)[COUNTS].sum()
    daily["streak"] = df.groupby(["team_id", "game_date"])["result"].last().pipe(
        lambda s: s.groupby(level="team_id", group_keys=False).apply(streak_after)
    ).to_numpy()
    grouped = daily.groupby("team_id")
    out = [daily[["team_id", "game_date", "streak"]]]
    for name, n in WINDOWS.items():
        if n is None:
            summed = grouped[COUNTS].cumsum()
        elif n == "ewm":
            summed = grouped[COUNTS].transform(lambda s: s.ewm(halflife=EWM_HALFLIFE).mean())
        else:
            summed = grouped[COUNTS].rolling(n, min_periods=1).sum().reset_index(level=0, drop=True)
        summed.columns = [f"{c}_{name}" for c in COUNTS]
        out.append(rates(summed, name))
    return pd.concat(out, axis=1)

## One row per game, as of the morning of the game

`merge_asof(..., allow_exact_matches=False)` picks each team's state after its last game on
a **strictly earlier** date.

In [ ]:
def as_of(state: pd.DataFrame, keys: pd.DataFrame, team_col: str, id_col: str = "game_id") -> pd.DataFrame:
    left = keys[[id_col, "game_date", team_col]].rename(columns={team_col: "team_id"})
    merged = pd.merge_asof(
        left.sort_values("game_date"),
        state.sort_values("game_date"),
        on="game_date",
        by="team_id",
        allow_exact_matches=False,  # strictly earlier dates only
    )
    return merged.set_index(id_col).drop(columns=["game_date", "team_id"])

### Strength of schedule

- **sos:** the average pre-game goal differential per game of the opponents a team has
  faced so far. Each opponent is counted as it stood on the day of that game, so the value
  itself uses no later information.
- **adj_goal_diff_pg:** season goal differential per game + sos. A +1.0 team that played a
  +0.5 schedule rates higher than a +1.0 team that played a −0.5 one.

In [ ]:
def schedule_state(team_games: pd.DataFrame, state: pd.DataFrame) -> pd.DataFrame:
    tg = team_games.reset_index(drop=True).copy()
    tg["row"] = np.arange(len(tg))
    opp_gd = as_of(state[["team_id", "game_date", "goal_diff_pg_season"]], tg, "opponent_team_id", "row")
    tg["opp_gd"] = opp_gd.loc[tg["row"], "goal_diff_pg_season"].fillna(0).to_numpy()  # unknown opponent = average
    daily = tg.groupby(["team_id", "game_date"], as_index=False).agg(s=("opp_gd", "sum"), n=("opp_gd", "size"))
    daily = daily.sort_values(["team_id", "game_date"])
    g = daily.groupby("team_id")
    daily["sos"] = g["s"].cumsum() / g["n"].cumsum()
    return daily[["team_id", "game_date", "sos"]]

### Likely starting goalie

The goalie with the most starts in the team's last 5 games before the date (ties go to the
most recent start), same rule as the dashboard. Their save % comes from all their games
before the date. If they have faced fewer than 100 shots, the team's season save % is used
instead.

In [ ]:
MIN_STARTER_SHOTS = 100


def starter_state(goalies: pd.DataFrame) -> pd.DataFrame:
    """Per team and date: the likely starter's save % going into the next game (after that date)."""
    rows = []
    starts = goalies[goalies["is_starter"]].sort_values(["team_id", "game_date", "game_id"])
    by_goalie = goalies.sort_values("game_date")
    for team, s in starts.groupby("team_id"):
        for date in s["game_date"].unique():
            last5 = s[s["game_date"] <= date].tail(5)
            counts = last5.groupby("player_id").agg(n=("game_id", "size"), last=("game_date", "max"))
            pid = counts.sort_values(["n", "last"], ascending=False).index[0]
            hist = by_goalie[(by_goalie["player_id"] == pid) & (by_goalie["game_date"] <= date)]
            sa = hist["shots_against"].sum()
            sv = hist["saves"].sum() / sa if sa >= MIN_STARTER_SHOTS else np.nan
            rows.append({"team_id": team, "game_date": date, "starter_save_pct": sv})
    return pd.DataFrame(rows)

In [ ]:
def pregame_table(team_games: pd.DataFrame, goalies: pd.DataFrame | None = None) -> pd.DataFrame:
    state = team_state(team_games)
    state = state.merge(schedule_state(team_games, state), on=["team_id", "game_date"], how="left")
    if goalies is not None and len(goalies):
        state = state.merge(starter_state(goalies), on=["team_id", "game_date"], how="left")
    else:
        state["starter_save_pct"] = np.nan
    state["starter_save_pct"] = state["starter_save_pct"].fillna(state["save_pct_season"])
    state["adj_goal_diff_pg"] = state["goal_diff_pg_season"] + state["sos"]
    state["form_delta"] = state["win_pct_last5"] - state["win_pct_season"]

    home = team_games[team_games["is_home"]].rename(columns={"team_id": "home_id", "opponent_team_id": "away_id"})
    games = home[["game_id", "season", "game_date", "home_id", "away_id", "is_neutral", "result"]]
    h = as_of(state, games, "home_id").add_prefix("home_")
    a = as_of(state, games, "away_id").add_prefix("away_")
    out = games.set_index("game_id").join(h).join(a)
    out["home_gp"] = out["home_gp_season"].fillna(0).astype(int)
    out["away_gp"] = out["away_gp_season"].fillna(0).astype(int)
    out["home_ice"] = (~out["is_neutral"]).astype(int)  # 1 = home game, 0 = neutral site

    diffs = {}
    for stem in TIER1:
        for w in WINDOWS:
            diffs[f"diff_{stem}_{w}"] = out[f"home_{stem}_{w}"] - out[f"away_{stem}_{w}"]
    for stem in TIER2:
        diffs[f"diff_{stem}_season"] = out[f"home_{stem}_season"] - out[f"away_{stem}_season"]
    for stem in EXTRA:
        diffs[f"diff_{stem}"] = out[f"home_{stem}"] - out[f"away_{stem}"]
    out = pd.concat([out, pd.DataFrame(diffs)], axis=1)
    out["home_win"] = out["result"].map({"W": 1, "L": 0})  # NaN for ties
    return out.reset_index().sort_values(["game_date", "game_id"]).reset_index(drop=True)


def model_rows(table: pd.DataFrame, min_gp: int = 3) -> pd.DataFrame:
    """Games used for modeling: no ties, and both teams past the warm-up."""
    keep = table["home_win"].notna() & (table["home_gp"] >= min_gp) & (table["away_gp"] >= min_gp)
    out = table[keep].copy()
    out["home_win"] = out["home_win"].astype(int)
    return out.reset_index(drop=True)


def feature_columns(table: pd.DataFrame) -> list[str]:
    return [c for c in table.columns if c.startswith("diff_")]

## Leakage checks

A small made-up season with known answers. Every `assert` must pass before the real data is
used. Team 3 plays two games on Oct 10, so a same-day leak would show up.

In [ ]:
def _tg(game_id, date, team, opp, home, gf, ga, sf=30, sa=30):
    result = "W" if gf > ga else "L" if gf < ga else "T"
    return {
        "season": "2025-26", "game_id": game_id, "game_date": pd.Timestamp(date),
        "team_id": team, "opponent_team_id": opp, "is_home": home, "is_neutral": False,
        "result": result, "goals_for": gf, "goals_against": ga, "shots_for": sf,
        "shots_against": sa, "pp_goals": 1, "pp_opportunities": 4, "pp_goals_against": 1,
        "times_shorthanded": 4, "en_goals": 0, "pim": 6, "shots_p1": sf // 3,
        "shots_p2": sf // 3, "shots_p3": sf - 2 * (sf // 3),
    }


def _game(game_id, date, home, away, hg, ag, **kw):
    sf, sa = kw.get("sf", 30), kw.get("sa", 30)
    return [_tg(game_id, date, home, away, True, hg, ag, sf=sf, sa=sa),
            _tg(game_id, date, away, home, False, ag, hg, sf=sa, sa=sf)]


TOY = pd.DataFrame(
    _game("g1", "2025-10-03", 1, 2, 4, 1, sf=40, sa=20)
    + _game("g2", "2025-10-04", 2, 1, 2, 2)
    + _game("g3", "2025-10-10", 1, 3, 1, 5)
    + _game("g4", "2025-10-10", 3, 2, 0, 3)  # team 3's second game the same day
    + _game("g5", "2025-10-17", 3, 1, 2, 3)
)


def _goals(df):
    """A goal list matching the toy scores: every goal in period 1, home goals first."""
    rows = []
    for gid, g in df.groupby("game_id", sort=False):
        n = 0
        for _, r in g.sort_values("is_home", ascending=False).iterrows():
            for _ in range(r["goals_for"]):
                n += 1
                rows.append({"game_id": gid, "goal_number": n, "team_id": r["team_id"], "period": 1})
    return pd.DataFrame(rows, columns=["game_id", "goal_number", "team_id", "period"])


toy_tg = add_ingredients(TOY, _goals(TOY))
toy = pregame_table(toy_tg).set_index("game_id")

# 1. First game: no history
assert toy.loc["g1", "home_gp"] == 0 and pd.isna(toy.loc["g1", "diff_goal_diff_pg_season"])

# 2. Only earlier dates: team 1 before Oct 10 = W 4-1 (40-20 shots), T 2-2 (30-30)
assert toy.loc["g3", "home_gp"] == 2
assert abs(toy.loc["g3", "home_goal_diff_pg_season"] - 1.5) < 1e-9
assert abs(toy.loc["g3", "home_win_pct_season"] - 0.75) < 1e-9
assert abs(toy.loc["g3", "home_shot_share_season"] - 70 / 120) < 1e-9
assert toy.loc["g3", "home_streak"] == 0  # tied its last game

# 3. Same-day games don't see each other
assert toy.loc["g3", "away_gp"] == 0 and toy.loc["g4", "home_gp"] == 0

# 4. Changing a later game changes nothing earlier
changed = TOY.copy()
changed.loc[changed["game_id"] == "g5", ["goals_for", "goals_against"]] = [9, 0]
cols = [c for c in toy.columns if c.startswith(("home_", "away_", "diff_"))]
pd.testing.assert_frame_equal(
    toy.loc[["g1", "g2", "g3", "g4"], cols],
    pregame_table(add_ingredients(changed, _goals(TOY))).set_index("game_id").loc[["g1", "g2", "g3", "g4"], cols],
)

# 5. Windows, schedule and the home-minus-away difference
assert abs(toy.loc["g5", "away_goal_diff_pg_last3"] - (-1 / 3)) < 1e-9
assert abs(toy.loc["g5", "diff_goal_diff_pg_season"]
           - (toy.loc["g5", "home_goal_diff_pg_season"] - toy.loc["g5", "away_goal_diff_pg_season"])) < 1e-9
# team 2 before Oct 4 had played only team 1, which was 0 GP that morning -> schedule 0
assert toy.loc["g2", "home_sos"] == 0
# close-game shots: every toy goal is in period 1, so g1 (4-1) is not close from period 2 on
assert toy_tg.query("game_id == 'g1' and is_home")["close_sf"].item() == 40 // 3

# 6. model_rows drops ties and warm-up games
assert toy.loc["g2", "home_win"] != toy.loc["g2", "home_win"]  # NaN for the tie
assert set(model_rows(pregame_table(toy_tg), min_gp=1)["game_id"]) == {"g5"}
print("All leakage checks passed.")

## Build the table for the season

Prints counts only. The table stays in memory for the notebooks that `%run` this one
(`team_games`, `goalies`, `table`, `rows`).

In [ ]:
with psycopg.connect(settings.database_url) as conn:
    team_games, goalies = load(conn, SEASON)

table = pregame_table(team_games, goalies)
rows = model_rows(table)
FEATURES = feature_columns(table)
print(f"{SEASON}: {len(table)} games, {int(table['home_win'].isna().sum())} ties")
print(f"After warm-up (both teams >= 3 GP) and dropping ties: {len(rows)} games")
print(f"{len(FEATURES)} diff_ features; missing values after warm-up: {int(rows[FEATURES].isna().sum().sum())}")